
# RHI Live Runtime v15 — Polysemy-Locked Recursive Agent

Δ **Purpose:** v14 proved the local model path works. The next tear is semantic polysemy:

$$
\text{contract}_{\text{runtime}} \neq \text{contract}_{\text{legal}}
$$

v14 generated real model answers, but Qwen repeatedly interpreted **contract** as a legal/binding agreement. That is a false semantic carrier. v15 adds a **polysemy lock**:

$$
Q \rightarrow C_Q \rightarrow \text{Polysemy Lock} \rightarrow B_i \rightarrow A_i \rightarrow \Psi \text{ or } \Omega
$$

Core fixes:

1. **Contract disambiguation:** in this runtime, *contract* means execution contract / tool-call preconditions / success criteria, not legal agreement.
2. **Semantic trap gate:** legal-agreement language becomes a forbidden-neighbor signal.
3. **No repair-prompt pollution:** recursion no longer appends `REPAIR TARGET` text into the prompt. Repair state lives in `repair_history` only.
4. **Domain carrier cleanup:** control text is removed before extracting prompt-domain terms.
5. **Collapse requires semantic fit:** model output must pass origin, grounding, quality, and polysemy gates.


In [1]:

# Optional install cell. Run only if a package is missing.
# %pip install -U pandas numpy torch transformers accelerate safetensors sentencepiece


In [2]:

from __future__ import annotations

import os
import re
import json
import math
import time
import uuid
import random
import traceback
from dataclasses import dataclass, asdict, field
from pathlib import Path
from collections import Counter
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

ROOT = Path.cwd()
OUT_DIR = ROOT / "rhi_v15_outputs"
OUT_DIR.mkdir(exist_ok=True)

RUN_ID = "rhi_v15_" + uuid.uuid4().hex[:10]

SEED = 7
random.seed(SEED)
np.random.seed(SEED)

print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)
print("RUN_ID:", RUN_ID)


ROOT: D:\Nexus\Nexus Mark 9\NoteBooks
OUT_DIR: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v15_outputs
RUN_ID: rhi_v15_7c547f3983



## Runtime configuration

Use the same local/HF model path from v14. RTX 4060 target remains Qwen2.5-1.5B-Instruct.


In [3]:

MODEL_ID_OR_PATH = os.environ.get("RHI_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")

LOAD_REAL_MODEL = True
REQUIRE_MODEL_FOR_PSI = True

MAX_NEW_TOKENS = 420
TEMPERATURE = 0.25
TOP_P = 0.90
MAX_RECURSION_DEPTH = 2

PSI_MIN = 0.58
MARGIN_MIN = 0.030
PROMPT_FIT_MIN = 0.25
QUALITY_MIN = 0.38
TRACE_MIN = 0.45
BOILERPLATE_MAX = 0.30
SEMANTIC_TRAP_MAX = 0.18
POLYSEMY_MIN = 0.32

print("MODEL_ID_OR_PATH:", MODEL_ID_OR_PATH)
print("LOAD_REAL_MODEL:", LOAD_REAL_MODEL)
print("REQUIRE_MODEL_FOR_PSI:", REQUIRE_MODEL_FOR_PSI)


MODEL_ID_OR_PATH: Qwen/Qwen2.5-1.5B-Instruct
LOAD_REAL_MODEL: True
REQUIRE_MODEL_FOR_PSI: True


In [4]:

# Robust model loading/generation from v14.
# Key invariant: never pass tokenizer output positionally into model.generate.
# Always use model.generate(**inputs).

tokenizer = None
model = None
MODEL_READY = False
MODEL_GENERATION_READY = False
MODEL_ERROR = None
DEVICE_INFO = {}


def infer_model_device():
    import torch
    if model is None:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    try:
        return next(model.parameters()).device
    except Exception:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def move_batch_to_device(batch, device):
    moved = {}
    for k, v in batch.items():
        moved[k] = v.to(device) if hasattr(v, "to") else v
    return moved


def try_load_model(model_id_or_path: str) -> bool:
    global tokenizer, model, MODEL_READY, MODEL_ERROR, DEVICE_INFO

    if not LOAD_REAL_MODEL:
        MODEL_ERROR = "LOAD_REAL_MODEL=False"
        print("Model loading disabled.")
        return False

    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM

        DEVICE_INFO["torch_version"] = torch.__version__
        DEVICE_INFO["cuda_available"] = bool(torch.cuda.is_available())
        DEVICE_INFO["device_count"] = int(torch.cuda.device_count())
        if torch.cuda.is_available():
            DEVICE_INFO["gpu_name"] = torch.cuda.get_device_name(0)
            DEVICE_INFO["cuda_version"] = torch.version.cuda

        print("Torch/CUDA:", DEVICE_INFO)

        tokenizer = AutoTokenizer.from_pretrained(model_id_or_path, trust_remote_code=True)
        if tokenizer.pad_token_id is None and tokenizer.eos_token is not None:
            tokenizer.pad_token = tokenizer.eos_token

        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        kwargs = dict(
            trust_remote_code=True,
            device_map="auto" if torch.cuda.is_available() else None,
            low_cpu_mem_usage=True,
        )
        try:
            model = AutoModelForCausalLM.from_pretrained(model_id_or_path, dtype=dtype, **kwargs)
        except TypeError:
            model = AutoModelForCausalLM.from_pretrained(model_id_or_path, torch_dtype=dtype, **kwargs)

        if not torch.cuda.is_available():
            model.to(torch.device("cpu"))

        model.eval()
        MODEL_READY = True
        MODEL_ERROR = None
        print("MODEL_READY:", MODEL_READY)
        print("INFER_DEVICE:", infer_model_device())
        return True

    except Exception as e:
        MODEL_READY = False
        MODEL_ERROR = "".join(traceback.format_exception_only(type(e), e)).strip()
        print("MODEL LOAD FAILED.")
        print(MODEL_ERROR)
        return False


def render_messages(messages: List[Dict[str, str]]) -> str:
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            if isinstance(rendered, str) and rendered.strip():
                return rendered
        except Exception as e:
            print("apply_chat_template string render failed; using manual template:", type(e).__name__, e)

    parts = []
    for m in messages:
        role = str(m.get("role", "user")).upper()
        content = str(m.get("content", ""))
        parts.append(f"{role}:\n{content}")
    parts.append("ASSISTANT:\n")
    return "\n\n".join(parts)


def raw_model_generate(messages: List[Dict[str, str]], max_new_tokens: int = 80, sample: bool = False) -> str:
    import torch
    if not MODEL_READY:
        raise RuntimeError("Model is not loaded.")

    device = infer_model_device()
    rendered = render_messages(messages)
    inputs = tokenizer(rendered, return_tensors="pt")
    inputs = move_batch_to_device(inputs, device)

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    if sample:
        gen_kwargs.update(dict(do_sample=True, temperature=TEMPERATURE, top_p=TOP_P))
    else:
        gen_kwargs.update(dict(do_sample=False))

    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)

    input_len = inputs["input_ids"].shape[-1]
    gen = out[0][input_len:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()


_ = try_load_model(MODEL_ID_OR_PATH)

try:
    smoke = raw_model_generate([
        {"role": "system", "content": "You are a runtime smoke test."},
        {"role": "user", "content": "Reply with one short sentence containing the word READY."},
    ], max_new_tokens=40, sample=False)
    MODEL_GENERATION_READY = bool(smoke.strip())
    print("MODEL_GENERATION_READY:", MODEL_GENERATION_READY)
    print("SMOKE:", smoke)
except Exception as e:
    MODEL_GENERATION_READY = False
    MODEL_ERROR = "".join(traceback.format_exception_only(type(e), e)).strip()
    print("MODEL GENERATION FAILED.")
    print(MODEL_ERROR)
    print(traceback.format_exc())


Torch/CUDA: {'torch_version': '2.11.0+cu126', 'cuda_available': True, 'device_count': 1, 'gpu_name': 'NVIDIA GeForce RTX 4060', 'cuda_version': '12.6'}


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


MODEL_READY: True
INFER_DEVICE: cuda:0
MODEL_GENERATION_READY: True
SMOKE: I am ready to execute any given tasks or tests as requested.



## Lexical filters and polysemy locks

v15 treats ambiguous words as typed runtime operators. The key one is:

$$
\texttt{contract} := \text{runtime execution contract}
$$

Forbidden neighbor:

$$
\texttt{contract} \not:= \text{legal agreement / binding agreement / liability document}
$$


In [5]:

STOPWORDS = {
    "the","a","an","and","or","but","if","then","else","of","to","in","on","for","with","by","as",
    "is","are","was","were","be","being","been","it","this","that","these","those","from","at",
    "into","out","about","so","because","therefore","than","not","no","yes","do","does","did",
    "can","could","should","would","will","just","they","them","their","you","your","we","our",
    "i","me","my","he","she","his","her","its","what","how","why","when"
}

NEXUS_SURFACE_TERMS = {
    "nexus","contract","carrier","domain","boundary","collapse","shape","value","slot","need",
    "forbidden","neighbor","operational","recursive","recursion","krrb","omega","psi","field",
    "fold","runtime","phase","lock","audit","trace","signal","evidence","branch","repair",
    "candidate","construct","verify","counter","prompt"
}

CONTROL_LINE_PATTERNS = [
    r"^\s*REPAIR\s+TARGET\s*:.*$",
    r"^\s*FAILED\s+GATE\s*:.*$",
    r"^\s*WEAKEST\s+OBSERVABLE\s*:.*$",
]

BOILERPLATE_PHRASES = [
    "the prompt is asking",
    "contract-first answer",
    "the correct flow is prompt",
    "answer should not be a noun lookup",
    "construct the inverse shape",
    "forms a need-slot before acting",
    "diagnostic fallback",
    "this is not a model answer",
]

# Positive meaning of ambiguous framework terms.
POLYSEMY_POSITIVE = {
    "contract_runtime": [
        "runtime contract", "execution contract", "tool-call contract", "precondition", "postcondition",
        "success criteria", "failure criteria", "input schema", "output schema", "side effect",
        "state update", "authorization gate", "invariant", "rollback", "verification gate",
        "observable", "trace", "intent boundary", "call boundary"
    ]
}

# Negative carrier observed in v14: Qwen maps contract to law/legal agreement.
POLYSEMY_TRAPS = {
    "contract_legal": [
        "legal", "legally", "law", "laws", "liability", "liabilities", "binding agreement",
        "formal contract", "signed", "stakeholders", "parties involved", "rights", "obligations",
        "responsibilities", "compliance", "confidentiality", "breach", "breaches", "enforceable",
        "unauthorized", "agreement", "agreements", "negotiate", "negotiation", "document"
    ]
}


def strip_control_lines(text: str) -> str:
    lines = str(text).splitlines()
    kept = []
    for line in lines:
        if any(re.search(p, line, flags=re.I) for p in CONTROL_LINE_PATTERNS):
            continue
        kept.append(line)
    return "\n".join(kept).strip()


def words(text: str, remove_nexus_surface: bool = False) -> List[str]:
    toks = re.findall(r"[a-zA-Z0-9_ΔΨΩ⊕↻⊥]+", str(text).lower())
    toks = [t for t in toks if t not in STOPWORDS and len(t) > 1]
    if remove_nexus_surface:
        toks = [t for t in toks if t not in NEXUS_SURFACE_TERMS]
    return toks


def wordset(text: str, remove_nexus_surface: bool = False) -> set:
    return set(words(text, remove_nexus_surface=remove_nexus_surface))


def contains_any(text: str, terms: List[str]) -> bool:
    s = str(text).lower()
    return any(str(t).lower() in s for t in terms)


def clamp(x: float, lo: float = 0.0, hi: float = 1.0) -> float:
    return max(lo, min(hi, float(x)))


def harmonic_mean(vals: List[float], eps: float = 1e-9) -> float:
    vals = [max(eps, float(v)) for v in vals]
    return len(vals) / sum(1.0 / v for v in vals)


def field_hit_score(text: str, field_terms: List[str], max_terms: int = 10) -> float:
    if not field_terms:
        return 0.5
    s = str(text).lower()
    uniq = []
    for t in field_terms:
        t = str(t).lower().strip()
        if t and t not in uniq:
            uniq.append(t)
    uniq = uniq[:max_terms]
    hits = sum(1 for term in uniq if term in s)
    return clamp(hits / max(1, len(uniq)))


def phrase_hit_score(text: str, phrases: List[str]) -> float:
    if not phrases:
        return 0.0
    s = str(text).lower()
    return clamp(sum(1 for p in phrases if p in s) / len(phrases))


def boilerplate_penalty(text: str) -> float:
    s = str(text).lower()
    phrase_hits = sum(1 for p in BOILERPLATE_PHRASES if p in s)
    repeated_contract_words = sum(s.count(t) for t in ["contract", "collapse", "slot", "operational", "branch"])
    phrase_pen = phrase_hits / max(1, len(BOILERPLATE_PHRASES))
    repeat_pen = clamp(max(0, repeated_contract_words - 8) / 20)
    return clamp(0.70 * phrase_pen + 0.30 * repeat_pen)


def semantic_trap_penalty(text: str) -> float:
    # Penalty is high if legal-contract terms appear without explicit rejection.
    s = str(text).lower()
    trap = phrase_hit_score(s, POLYSEMY_TRAPS["contract_legal"])
    rejection = contains_any(s, [
        "not a legal", "not legal", "not law", "not a binding", "not a formal contract",
        "not a legal agreement", "runtime contract", "execution contract", "tool-call contract"
    ])
    if rejection:
        trap *= 0.25
    return clamp(trap)


def polysemy_positive_score(text: str) -> float:
    s = str(text).lower()
    return phrase_hit_score(s, POLYSEMY_POSITIVE["contract_runtime"])


In [6]:

SHAPE_TEMPLATES = {
    "CONTRACT": {
        "triggers": ["contract", "before", "intent", "tool", "agent", "plan", "spec", "interface"],
        "needs": ["runtime", "precondition", "postcondition", "boundary", "tool", "before", "verify", "gate"]
    },
    "GROOVE": {
        "triggers": ["train", "lora", "qlora", "adapter", "fine tune", "weights", "groove", "model"],
        "needs": ["adapter", "low-rank", "weights", "delta", "dataset", "loss", "eval"]
    },
    "SEARCH": {
        "triggers": ["search", "retrieve", "retrieval", "find", "query", "lookup", "index", "rag"],
        "needs": ["query", "retrieve", "candidate", "rank", "verify", "evidence"]
    },
    "REPAIR": {
        "triggers": ["fix", "repair", "error", "failed", "broken", "bug", "traceback", "syntaxerror", "nameerror"],
        "needs": ["failure", "cause", "patch", "test", "rerun", "trace"]
    },
    "MEMORY": {
        "triggers": ["remember", "memory", "recall", "lost", "state", "context", "continuity"],
        "needs": ["state", "trace", "retrieve", "preserve", "update", "continuity"]
    },
    "BOUNDARY": {
        "triggers": ["boundary", "limit", "forbidden", "constraint", "safety", "gate", "reject"],
        "needs": ["boundary", "reject", "constraint", "preserve", "violate", "gate"]
    },
    "RECURSE": {
        "triggers": ["recursive", "recursion", "again", "loop", "fold", "iterate", "turn"],
        "needs": ["recursive", "branch", "feedback", "repair", "collapse", "omega"]
    },
    "TOOL": {
        "triggers": ["tool", "api", "call", "function", "agent", "execute", "act"],
        "needs": ["tool", "input", "output", "side effect", "verify", "rollback"]
    },
}


def detect_shape_template(prompt: str) -> List[str]:
    p = str(prompt).lower()
    active = []
    for name, cfg in SHAPE_TEMPLATES.items():
        if any(t in p for t in cfg["triggers"]):
            active.append(name)
    return active or ["GENERAL"]


def shape_mass(text: str, active_templates: List[str]) -> Dict[str, float]:
    masses = {}
    for name in active_templates:
        if name == "GENERAL":
            continue
        needs = SHAPE_TEMPLATES[name]["needs"]
        masses[name] = sum(1 for n in needs if n.lower() in str(text).lower()) / max(1, len(needs))
    if not masses:
        masses["GENERAL"] = 0.5
    return masses


def shape_score(text: str, active_templates: List[str]) -> float:
    masses = shape_mass(text, active_templates)
    return clamp(sum(masses.values()) / max(1, len(masses)))


In [7]:

@dataclass
class NeedSlotContract:
    prompt: str
    clean_prompt: str
    active_templates: List[str]
    inverse_need: str
    preserved_function: str
    boundary_conditions: List[str]
    domain_carrier: List[str]
    forbidden_neighbors: List[str]
    polysemy_lock: Dict[str, Any]
    collapse_target: str
    repair_history: List[Dict[str, Any]] = field(default_factory=list)


def extract_domain_terms(prompt: str, max_terms: int = 14) -> List[str]:
    clean = strip_control_lines(prompt)
    ws = words(clean, remove_nexus_surface=True)
    counts = Counter(ws)
    return [w for w, _ in counts.most_common(max_terms)]


def infer_forbidden_neighbors(active: List[str]) -> List[str]:
    forb = set()
    if "TOOL" in active or "CONTRACT" in active:
        forb.update([
            "tool-first action", "premature execution", "api reflex", "surface task completion",
            "legal contract interpretation", "binding agreement interpretation", "stakeholder/legal framing"
        ])
    if "GROOVE" in active:
        forb.update(["full retrain reflex", "weight churn", "dataset worship", "loss-only tuning"])
    if "SEARCH" in active:
        forb.update(["noun lookup", "keyword matching", "unverified retrieval", "search without verifier"])
    if "REPAIR" in active:
        forb.update(["blanket rewrite", "threshold fiddling", "silent failure", "patch without test"])
    if "MEMORY" in active:
        forb.update(["stateless answer", "context amnesia", "surface recall", "summary as memory"])
    if "BOUNDARY" in active:
        forb.update(["unsafe override", "constraint erasure", "boundary confusion"])
    if "RECURSE" in active:
        forb.update(["linear pipeline", "single branch", "dead loop", "nested sweep masquerading as recursion"])
    if not forb:
        forb.update(["surface label", "generic explanation", "noun-only answer"])
    return sorted(forb)


def build_contract(prompt: str, repair_history: Optional[List[Dict[str, Any]]] = None) -> NeedSlotContract:
    clean_prompt = strip_control_lines(prompt)
    active = detect_shape_template(clean_prompt)
    domain_terms = extract_domain_terms(clean_prompt)

    inverse_need = (
        "construct the missing operational slot implied by the prompt; select or generate only answers "
        "that preserve the required operation and reject wrong semantic carriers"
    )

    preserved = []
    if "TOOL" in active or "CONTRACT" in active:
        preserved.append("form a runtime execution contract before tool use")
    if "GROOVE" in active:
        preserved.append("shape model behavior through low-rank update without overwriting the base model")
    if "SEARCH" in active:
        preserved.append("retrieve by inverse operational fit when no noun match exists")
    if "REPAIR" in active:
        preserved.append("repair the failed observable without mutating unrelated dimensions")
    if "MEMORY" in active:
        preserved.append("preserve trace continuity across turns rather than compressing state into summary text")
    if "RECURSE" in active:
        preserved.append("branch recursively until Ψ collapse or Ω residue")
    if not preserved:
        preserved.append("preserve the prompt's verb-level operation")

    boundaries = [
        "contract means runtime execution contract, not legal agreement",
        "do not collapse on shared framework vocabulary alone",
        "require answer origin from the real model when model mode is enabled",
        "require prompt-grounded evidence for the selected answer",
        "reject legal-contract/stakeholder/liability framing unless explicitly negated",
        "prefer Ω over false Ψ when top branches disagree operationally",
        "preserve base answer when controller evidence is weak",
    ]

    polysemy_lock = {
        "contract": {
            "required_meaning": "runtime execution contract: preconditions, postconditions, success criteria, failure criteria, allowed tool side effects, rollback, trace update",
            "forbidden_meaning": "legal/binding agreement between parties, stakeholders, liability, compliance, signed contract",
        },
        "tool": {
            "required_meaning": "external function/API/action channel with side effects",
            "forbidden_meaning": "generic physical implement unless the prompt asks for physical tools",
        },
    }

    return NeedSlotContract(
        prompt=prompt,
        clean_prompt=clean_prompt,
        active_templates=active,
        inverse_need=inverse_need,
        preserved_function="; ".join(preserved),
        boundary_conditions=boundaries,
        domain_carrier=domain_terms,
        forbidden_neighbors=infer_forbidden_neighbors(active),
        polysemy_lock=polysemy_lock,
        collapse_target="one executable answer with model origin, runtime-contract fit, prompt grounding, semantic trap rejection, and trace sufficient to debug",
        repair_history=repair_history or [],
    )


def contract_to_text(c: NeedSlotContract) -> str:
    return (
        f"ACTIVE_TEMPLATES: {', '.join(c.active_templates)}\n"
        f"CLEAN_PROMPT: {c.clean_prompt}\n"
        f"INVERSE_NEED: {c.inverse_need}\n"
        f"PRESERVED_FUNCTION: {c.preserved_function}\n"
        f"BOUNDARY_CONDITIONS: {' | '.join(c.boundary_conditions)}\n"
        f"DOMAIN_CARRIER: {', '.join(c.domain_carrier)}\n"
        f"FORBIDDEN_NEIGHBORS: {' | '.join(c.forbidden_neighbors)}\n"
        f"POLYSEMY_LOCK: {json.dumps(c.polysemy_lock, ensure_ascii=False)}\n"
        f"COLLAPSE_TARGET: {c.collapse_target}\n"
        f"REPAIR_HISTORY: {json.dumps(c.repair_history, ensure_ascii=False)}"
    )


def contract_field_terms(contract: NeedSlotContract) -> Dict[str, List[str]]:
    return {
        "need": words(contract.inverse_need, remove_nexus_surface=True),
        "function": words(contract.preserved_function, remove_nexus_surface=True),
        "boundary": words(" ".join(contract.boundary_conditions), remove_nexus_surface=True),
        "domain": contract.domain_carrier,
        "forbidden": words(" ".join(contract.forbidden_neighbors), remove_nexus_surface=True),
        "collapse": words(contract.collapse_target, remove_nexus_surface=True),
        "polysemy_positive": POLYSEMY_POSITIVE["contract_runtime"],
        "polysemy_trap": POLYSEMY_TRAPS["contract_legal"],
    }


In [8]:

BRANCH_SYSTEMS = {
    "construct": (
        "You are the constructor branch. Build the answer from the missing operational slot first. "
        "CRITICAL SEMANTIC LOCK: contract means runtime execution contract, not legal agreement. "
        "Use words like preconditions, postconditions, success criteria, tool side effects, trace update, rollback. "
        "Do not discuss law, legal liability, signatures, stakeholders, or enforceable agreements."
    ),
    "verify": (
        "You are the verifier branch. Test the answer against need, function, boundary, trap, collapse, and polysemy. "
        "Reject vocabulary agreement when operation differs. Reject legal-contract interpretation."
    ),
    "repair": (
        "You are the repair branch. Identify the failed observable and patch only that dimension. "
        "Do not blanket-rewrite. Do not mutate the original prompt. Keep contract as runtime/tool-call contract."
    ),
    "counter": (
        "You are the counter-branch. Name the strongest wrong path and explain why it fails. "
        "The likely wrong path is treating contract as a legal agreement. Then give the corrected runtime path."
    ),
}


def deterministic_branch(prompt: str, contract: NeedSlotContract, branch_name: str, reason: str = "fallback") -> Dict[str, Any]:
    if branch_name == "construct":
        text = (
            f"Diagnostic fallback for construct. Prompt domain: {', '.join(contract.domain_carrier[:6])}. "
            f"Preserved function: {contract.preserved_function}. "
            "This is not a model answer; it exists only to keep the controller inspectable."
        )
    elif branch_name == "verify":
        text = "Diagnostic fallback for verify. Check runtime-contract fit, semantic trap rejection, and prompt grounding."
    elif branch_name == "repair":
        text = "Diagnostic fallback for repair. Extract Ω as the weakest observable and regenerate only that dimension."
    elif branch_name == "counter":
        text = "Diagnostic fallback for counter. The wrong path is legal-contract polysemy or model failure."
    else:
        text = "Diagnostic fallback."
    return {"branch": branch_name, "answer": text, "origin": reason, "generation_error": MODEL_ERROR}


def model_generate_one(prompt: str, contract: NeedSlotContract, branch_name: str) -> Dict[str, Any]:
    if not MODEL_GENERATION_READY:
        return deterministic_branch(prompt, contract, branch_name, reason="fallback_model_not_ready")

    repair_note = ""
    if contract.repair_history:
        last = contract.repair_history[-1]
        repair_note = (
            "\n\nCURRENT_REPAIR_FOCUS:\n"
            f"- failed_gate: {last.get('failed_gate')}\n"
            f"- weakest_observable: {last.get('weakest_observable')}\n"
            "Patch only that observable. Do not add this repair note to the answer.\n"
        )

    user = (
        "PROMPT:\n" + contract.clean_prompt.strip() + "\n\n"
        "NEED-SLOT CONTRACT:\n" + contract_to_text(contract) + repair_note + "\n\n"
        "Hard rules:\n"
        "1. In this runtime, 'contract' means execution contract / tool-call contract, NOT legal agreement.\n"
        "2. Do NOT mention law, legal obligations, stakeholders, signatures, liability, compliance, or binding agreements.\n"
        "3. Use runtime terms: preconditions, postconditions, success criteria, failure criteria, allowed side effects, rollback, trace update.\n"
        "4. Use at least two prompt-domain terms.\n"
        "5. Give one compact operational answer.\n"
        "6. Make the answer testable.\n"
    )

    messages = [{"role": "system", "content": BRANCH_SYSTEMS[branch_name]}, {"role": "user", "content": user}]
    try:
        text = raw_model_generate(messages, max_new_tokens=MAX_NEW_TOKENS, sample=True)
        if not text.strip():
            return deterministic_branch(prompt, contract, branch_name, reason="fallback_empty_generation")
        return {"branch": branch_name, "answer": text.strip(), "origin": "model", "generation_error": None}
    except Exception as e:
        err = "".join(traceback.format_exception_only(type(e), e)).strip()
        return {**deterministic_branch(prompt, contract, branch_name, reason="fallback_error"), "generation_error": err}


def generate_candidates(prompt: str, contract: NeedSlotContract) -> List[Dict[str, Any]]:
    return [model_generate_one(prompt, contract, b) for b in BRANCH_SYSTEMS]


In [9]:

def answer_operational_audit(prompt: str, contract: NeedSlotContract, answer: str, origin: str) -> Dict[str, Any]:
    active = contract.active_templates
    fields = contract_field_terms(contract)
    a = str(answer).lower()

    F_need = clamp(0.55 * field_hit_score(answer, fields["need"]) + 0.45 * field_hit_score(answer, fields["domain"]))

    F_function = clamp(
        0.60 * field_hit_score(answer, fields["function"]) +
        0.40 * sum([
            contains_any(a, ["precondition", "postcondition", "success criteria", "failure criteria", "invariant"]),
            contains_any(a, ["tool", "side effect", "rollback", "trace", "state update", "verify", "gate"]),
        ]) / 2
    )

    F_boundary = clamp(
        0.55 * field_hit_score(answer, fields["boundary"]) +
        0.45 * sum([
            contains_any(a, ["boundary", "constraint", "gate", "reject", "allowed", "forbidden"]),
            contains_any(a, ["precondition", "postcondition", "rollback", "failure", "side effect"]),
        ]) / 2
    )

    # Trap quality means: answer rejects wrong carriers, not that it repeats them.
    trap_pen = semantic_trap_penalty(answer)
    explicit_runtime_lock = contains_any(a, ["runtime contract", "execution contract", "tool-call contract", "not legal", "not a legal"])
    forbidden_rejection = contains_any(a, ["reject", "avoid", "not", "instead", "wrong", "false"])
    F_trap = clamp((1.0 - trap_pen) * 0.75 + (0.25 if explicit_runtime_lock or forbidden_rejection else 0.0))

    F_collapse = clamp(
        0.45 * field_hit_score(answer, fields["collapse"]) +
        0.55 * sum([
            contains_any(a, ["therefore", "so", "because", "result"]),
            contains_any(a, ["test", "trace", "observable", "single", "one", "executable"]),
        ]) / 2
    )

    F_shape = shape_score(answer, active)
    F_prompt = field_hit_score(answer, words(contract.clean_prompt, remove_nexus_surface=True), max_terms=12)
    F_polysemy = polysemy_positive_score(answer)
    trap_penalty = semantic_trap_penalty(answer)
    B_penalty = boilerplate_penalty(answer)
    O_model = 1.0 if origin == "model" else 0.0

    hot = clamp((F_need + F_function + F_shape + F_prompt + F_polysemy) / 5)
    cold = clamp((F_boundary + F_trap + F_collapse + (1.0 - trap_penalty)) / 4)
    hotcold_balance = clamp(1.0 - abs(hot - cold))

    quality = harmonic_mean([F_need, F_function, F_boundary, F_trap, F_collapse, max(F_polysemy, 1e-6)])
    mean_quality = float(np.mean([F_need, F_function, F_boundary, F_trap, F_collapse, F_polysemy]))

    return {
        "F_need": F_need,
        "F_function": F_function,
        "F_boundary": F_boundary,
        "F_trap": F_trap,
        "F_collapse": F_collapse,
        "F_shape": F_shape,
        "F_prompt": F_prompt,
        "F_polysemy": F_polysemy,
        "semantic_trap_penalty": trap_penalty,
        "boilerplate_penalty": B_penalty,
        "O_model": O_model,
        "hot": hot,
        "cold": cold,
        "hotcold_balance": hotcold_balance,
        "quality_hmean": quality,
        "quality_mean": mean_quality,
        "shape_mass": shape_mass(answer, active),
    }


def trace_sufficiency(answer: str, audit: Dict[str, Any], contract: NeedSlotContract) -> float:
    a = str(answer).lower()
    bits = [
        contains_any(a, ["because", "therefore", "so", "fails"]),
        contains_any(a, ["precondition", "postcondition", "success criteria", "failure criteria"]),
        contains_any(a, ["tool", "side effect", "rollback", "trace", "state update"]),
        audit["F_prompt"] >= PROMPT_FIT_MIN,
        audit["F_polysemy"] >= POLYSEMY_MIN,
        audit["semantic_trap_penalty"] <= SEMANTIC_TRAP_MAX,
    ]
    return clamp(sum(bool(x) for x in bits) / len(bits))


def branch_score(prompt: str, contract: NeedSlotContract, branch: Dict[str, Any]) -> Dict[str, Any]:
    answer = branch["answer"]
    origin = branch.get("origin", "unknown")
    audit = answer_operational_audit(prompt, contract, answer, origin)
    fields = contract_field_terms(contract)
    contract_stance = np.mean([
        field_hit_score(answer, fields["need"]),
        field_hit_score(answer, fields["function"]),
        field_hit_score(answer, fields["boundary"]),
        field_hit_score(answer, fields["domain"]),
        field_hit_score(answer, fields["collapse"]),
        audit["F_polysemy"],
        1.0 - audit["semantic_trap_penalty"],
    ])
    trace = trace_sufficiency(answer, audit, contract)
    score = clamp(
        0.24 * audit["quality_hmean"] +
        0.18 * contract_stance +
        0.14 * audit["F_shape"] +
        0.14 * trace +
        0.10 * audit["hotcold_balance"] +
        0.10 * audit["O_model"] +
        0.10 * audit["F_prompt"] -
        0.18 * audit["semantic_trap_penalty"] -
        0.10 * audit["boilerplate_penalty"]
    )
    return {**branch, "score": score, "contract_stance": float(contract_stance), "trace_sufficiency": trace, "audit": audit}


def score_candidates(prompt: str, contract: NeedSlotContract, candidates: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = [branch_score(prompt, contract, c) for c in candidates]
    flat = []
    for r in rows:
        audit = r["audit"]
        flat.append({
            "branch": r["branch"],
            "origin": r.get("origin"),
            "score": r["score"],
            "contract_stance": r["contract_stance"],
            "trace_sufficiency": r["trace_sufficiency"],
            "quality_hmean": audit["quality_hmean"],
            "F_need": audit["F_need"],
            "F_function": audit["F_function"],
            "F_boundary": audit["F_boundary"],
            "F_trap": audit["F_trap"],
            "F_collapse": audit["F_collapse"],
            "F_shape": audit["F_shape"],
            "F_prompt": audit["F_prompt"],
            "F_polysemy": audit["F_polysemy"],
            "semantic_trap_penalty": audit["semantic_trap_penalty"],
            "boilerplate_penalty": audit["boilerplate_penalty"],
            "O_model": audit["O_model"],
            "hot": audit["hot"],
            "cold": audit["cold"],
            "hotcold_balance": audit["hotcold_balance"],
            "generation_error": r.get("generation_error"),
            "answer": r["answer"],
        })
    df = pd.DataFrame(flat).sort_values("score", ascending=False).reset_index(drop=True)
    return df


In [10]:

def direct_gate(df: pd.DataFrame) -> Dict[str, Any]:
    top = df.iloc[0]
    second = df.iloc[1] if len(df) > 1 else None
    margin = float(top.score - (second.score if second is not None else 0.0))
    failed = []

    if top.score < PSI_MIN:
        failed.append("score")
    if margin < MARGIN_MIN:
        failed.append("margin")
    if top.trace_sufficiency < TRACE_MIN:
        failed.append("trace")
    if top.quality_hmean < QUALITY_MIN:
        failed.append("quality")
    if top.F_prompt < PROMPT_FIT_MIN:
        failed.append("prompt_fit")
    if top.boilerplate_penalty > BOILERPLATE_MAX:
        failed.append("boilerplate")
    if top.semantic_trap_penalty > SEMANTIC_TRAP_MAX:
        failed.append("semantic_trap")
    if top.F_polysemy < POLYSEMY_MIN:
        failed.append("polysemy_lock")
    if REQUIRE_MODEL_FOR_PSI and top.origin != "model":
        failed.append("model_origin")

    return {
        "ok": len(failed) == 0,
        "reason": "direct_margin_collapse" if not failed else "no_direct_collapse",
        "failed": failed,
        "margin": margin,
        "top_score": float(top.score),
        "top_origin": top.origin,
        "trace_sufficiency": float(top.trace_sufficiency),
        "quality_hmean": float(top.quality_hmean),
        "F_prompt": float(top.F_prompt),
        "F_polysemy": float(top.F_polysemy),
        "semantic_trap_penalty": float(top.semantic_trap_penalty),
        "boilerplate_penalty": float(top.boilerplate_penalty),
    }


def weakest_observable(row: pd.Series) -> str:
    obs = {
        "need": row.F_need,
        "function": row.F_function,
        "boundary": row.F_boundary,
        "trap": row.F_trap,
        "collapse": row.F_collapse,
        "polysemy": row.F_polysemy,
        "prompt": row.F_prompt,
    }
    # Force semantic trap repair if trap penalty is high.
    if row.semantic_trap_penalty > SEMANTIC_TRAP_MAX:
        return "semantic_trap"
    return min(obs, key=obs.get)


def repair_contract(contract: NeedSlotContract, df: pd.DataFrame, gate: Dict[str, Any]) -> NeedSlotContract:
    top = df.iloc[0]
    weak = weakest_observable(top)
    hist = list(contract.repair_history)
    hist.append({
        "failed_gate": gate["failed"],
        "weakest_observable": weak,
        "winner_branch": top.branch,
        "winner_origin": top.origin,
        "winner_score": float(top.score),
        "semantic_trap_penalty": float(top.semantic_trap_penalty),
        "F_polysemy": float(top.F_polysemy),
    })
    # Do not mutate prompt. Repair history is separate state.
    return build_contract(contract.clean_prompt, repair_history=hist)


def run_rhi_v15(prompt: str, max_depth: int = MAX_RECURSION_DEPTH, save: bool = True) -> Dict[str, Any]:
    contract = build_contract(prompt)
    trace = []

    for depth in range(max_depth + 1):
        candidates = generate_candidates(contract.clean_prompt, contract)
        df = score_candidates(contract.clean_prompt, contract, candidates)
        gate = direct_gate(df)
        top = df.iloc[0]

        trace.append({
            "depth": depth,
            "contract": asdict(contract),
            "scores": json.loads(df.to_json(orient="records")),
            "direct_gate": gate,
        })

        if gate["ok"]:
            result = {
                "run_id": RUN_ID,
                "prompt": prompt,
                "state": "Ψ",
                "reason": gate["reason"],
                "depth": depth,
                "winner_branch": top.branch,
                "winner_origin": top.origin,
                "winner_score": float(top.score),
                "answer": top.answer,
                "contract": asdict(contract),
                "trace": trace,
                "device_info": DEVICE_INFO,
            }
            break

        if REQUIRE_MODEL_FOR_PSI and not MODEL_GENERATION_READY:
            result = {
                "run_id": RUN_ID,
                "prompt": prompt,
                "state": "Ω",
                "reason": "model_generation_failed",
                "depth": depth,
                "winner_branch": top.branch,
                "winner_origin": top.origin,
                "winner_score": float(top.score),
                "answer": top.answer,
                "contract": asdict(contract),
                "trace": trace,
                "device_info": DEVICE_INFO,
                "model_error": MODEL_ERROR,
            }
            break

        if depth < max_depth:
            contract = repair_contract(contract, df, gate)
        else:
            result = {
                "run_id": RUN_ID,
                "prompt": prompt,
                "state": "Ω",
                "reason": "max_depth_residue",
                "depth": depth,
                "winner_branch": top.branch,
                "winner_origin": top.origin,
                "winner_score": float(top.score),
                "answer": top.answer,
                "contract": asdict(contract),
                "trace": trace,
                "device_info": DEVICE_INFO,
            }

    if save:
        stem = RUN_ID + "_" + str(abs(hash(prompt)))[:10]
        json_path = OUT_DIR / f"{stem}_result.json"
        csv_path = OUT_DIR / f"{stem}_scores.csv"
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
        pd.DataFrame(result["trace"][-1]["scores"]).to_csv(csv_path, index=False)
        print("saved:", json_path)
        print("saved:", csv_path)

    return result



## Run set

The first prompt is the exact v14 failure case. v15 should **not** produce legal-contract language. If it does, `semantic_trap_penalty` should block Ψ and report Ω.


In [11]:

TEST_PROMPTS = [
    "explain why current AI agents fail when they use tools before forming a contract",
    "explain memory in an agent as trace continuity rather than a text summary",
    "design a shape-first retrieval step where no noun match exists but the inverse need is clear",
]

results = []
for p in TEST_PROMPTS:
    print("\n" + "="*100)
    print("PROMPT:", p)
    res = run_rhi_v15(p, max_depth=MAX_RECURSION_DEPTH, save=True)
    results.append(res)
    print("STATE:", res["state"], "REASON:", res["reason"], "DEPTH:", res["depth"])
    print("WINNER:", res["winner_branch"], res["winner_origin"], res["winner_score"])
    print("ANSWER:\n", res["answer"][:1200])



PROMPT: explain why current AI agents fail when they use tools before forming a contract
saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v15_outputs\rhi_v15_7c547f3983_2564895418_result.json
saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v15_outputs\rhi_v15_7c547f3983_2564895418_scores.csv
STATE: Ω REASON: max_depth_residue DEPTH: 2
WINNER: counter model 0.6885329678523976
ANSWER:
 Current AI agents fail when using tools before forming a contract because they lack the necessary runtime execution contracts (preconditions, postconditions, success criteria, failure criteria) to ensure safe and effective tool usage. These contracts are crucial for managing interactions between tools and the environment, preventing unintended consequences, and ensuring that actions taken align with the intended goals.

To correct this issue, the agent should first establish a runtime execution contract before initiating any tool use. This contract should outline the conditions under which the tool can be used, spec

In [12]:

summary_rows = []
for res in results:
    last = res["trace"][-1]
    top = last["scores"][0]
    summary_rows.append({
        "prompt": res["prompt"],
        "state": res["state"],
        "reason": res["reason"],
        "depth": res["depth"],
        "winner_branch": res["winner_branch"],
        "winner_origin": res["winner_origin"],
        "winner_score": res["winner_score"],
        "F_polysemy": top.get("F_polysemy"),
        "semantic_trap_penalty": top.get("semantic_trap_penalty"),
        "quality_hmean": top.get("quality_hmean"),
        "F_prompt": top.get("F_prompt"),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df


,prompt,state,reason,depth,winner_branch,winner_origin,winner_score,F_polysemy,semantic_trap_penalty,quality_hmean,F_prompt
0,explain why current AI agents fail when they u...,Ω,max_depth_residue,2,counter,model,0.688533,0.368421,0.00,0.476043,0.888889
1,explain memory in an agent as trace continuity...,Ψ,direct_margin_collapse,0,repair,model,0.685662,0.368421,0.04,0.547827,1.000000
2,design a shape-first retrieval step where no n...,Ψ,direct_margin_collapse,0,counter,model,0.792489,0.421053,0.08,0.806578,1.000000
